Amardeep Singh

In [2]:
# ================================
# 🔥 INSTALL & IMPORTS
# ================================
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

# ================================
# 🔧 TEXT GENERATION FUNCTION
# ================================
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


# ============================================================
# 🔹 COMPONENT I: PRODUCT REVIEW GENERATOR
# ============================================================

print("\n================ PRODUCT REVIEW GENERATOR ================\n")

# LOAD MODEL
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# BASELINE
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE OUTPUT ===")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")


# DATASET
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

# TOKENIZE
dataset = Dataset.from_dict({"text": corpus})

tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# TRAIN (✅ FIXED ARGUMENT)
args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    eval_strategy="epoch",   # ✅ FIXED HERE
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=collator
)

trainer.train()

# AFTER TRAINING
eval_res = trainer.evaluate()
print(f"\nPerplexity: {math.exp(eval_res['eval_loss']):.2f}")

print("\n=== FINE-TUNED OUTPUT ===")
for p in review_prompts:
    out = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Before: {baseline[p][:120]}")
    print(f"After : {out[:120]}")


# ============================================================
# 🔹 COMPONENT II: RECIPE GENERATOR
# ============================================================

print("\n================ RECIPE GENERATOR ================\n")

# LOAD NEW MODEL
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

# BASELINE
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare a chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===")
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline2[p]}")

# DATASET
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste.",
    "add tomato puree and cook on low heat until oil separates.",
    "add the marinated chicken and cook until fully done.",
    "finish with fresh cream and serve hot with naan or rice.",
    "for pasta carbonara boil spaghetti until al dente.",
    "fry pancetta until crispy and set aside.",
    "mix egg yolks parmesan cheese and black pepper.",
    "combine pasta with pancetta and egg mixture.",
    "serve immediately with extra cheese."
]

dataset2 = Dataset.from_dict({"text": recipes})

tok2 = dataset2.map(
    lambda x: tokenizer2(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split2 = tok2.train_test_split(test_size=0.15, seed=42)

collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

# TRAIN (✅ FIXED AGAIN)
args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    eval_strategy="epoch",   # ✅ FIXED
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=collator2
)

trainer2.train()

# AFTER TRAINING
eval2 = trainer2.evaluate()
print(f"\nPerplexity: {math.exp(eval2['eval_loss']):.2f}")

print("\n=== FINE-TUNED RECIPES ===")
for p in recipe_prompts:
    out = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt: {p}")
    print(f"Before: {baseline2[p][:120]}")
    print(f"After : {out[:120]}")


================ PRODUCT REVIEW GENERATOR ================



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== BASELINE OUTPUT ===

Prompt: This product is
Output: This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt: I bought this phone and
Output: I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 processor with the 8

Prompt

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken or other condiments with it, heat the oil in a large saucepan over medium heat. Stir in salt, the ground cinnamon and onion and cook for 2 minutes. Remove and let cool slightly.

Transfer to a baking sheet covered with plastic wrap and covered with foil.

Dissolve the butter in a large bowl with the sugar, cream, vanilla, salt and pepper to taste.

Serve hot with a scoop of cheese, garnish with the

Prompt: For pasta carbonara
Output: For pasta carbonara, the cheese is made in a pan and added to the pasta. The cheese is then sauteed in a sauce on the stove for about 8-10 minutes. Once it is done, the sauce is added to the pasta, and then the pasta is removed from the pan and placed in the refrigerator for 1-2 weeks.


If you are a chef, and you wish to add some fresh herbs, then you can prepare a fresh pasta in advance and refrigerate

Prompt: To prepare a chocolate cake
Output: To prepare a chocola

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,5.101429
2,No log,4.714139
3,No log,4.497822
4,No log,4.245408
5,3.546503,4.104166
6,3.546503,4.021955
7,3.546503,3.984658
8,3.546503,3.972932
9,3.546503,3.976558
10,1.977526,3.975077



Perplexity: 53.25

=== FINE-TUNED RECIPES ===

Prompt: To make butter chicken
Before: To make butter chicken or other condiments with it, heat the oil in a large saucepan over medium heat. Stir in salt, the
After : To make butter chicken and paratha with garam masala.

Mix yogurt in yogurt bowl with yogurt and fry onions in paneer un

Prompt: For pasta carbonara
Before: For pasta carbonara, the cheese is made in a pan and added to the pasta. The cheese is then sauteed in a sauce on the st
After : For pasta carbonara with olive oil and saute onion until golden brown on both sides. Add ginger garlic paste and curry p

Prompt: To prepare a chocolate cake
Before: To prepare a chocolate cake, please use the following recipes to prepare this cake; to create a frosting, please use the
After : To prepare a chocolate cake for later.

Preheat oven to 350 degrees F.

In a bowl, mash together the butter and eggs unt
